# Group Pipeline — Integrated Preprocessing
### IT2011 — Progress Review I: Data Preprocessing and EDA
**Group ID:** 2026-Y2-S1-[XX]
**Members:** [Member 1] · [Member 2] · [Member 3] · [Member 4] · [Member 5] · [Member 6]
**Assigned dataset:** Rotten Tomatoes Movie Review Dataset (Cornell)

This notebook integrates every member's individually-owned preprocessing technique into a
single, logically ordered pipeline, showing how the group's work fits together end to end.
Each stage below is attributed to the member who owns and can explain it in detail — see their
individual notebook in this folder for the full explanation, justification, and EDA chart.

**Pipeline order:** load → check quality (M1) → clean text (M2) → encode labels (M3) →
check outliers (M4) → vectorize + scale (M5) → select/reduce features (M6) → final output.


## 0. Load the Dataset

In [ ]:
!pip install -q datasets scikit-learn pandas matplotlib seaborn

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.decomposition import TruncatedSVD

sns.set_style("whitegrid")

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

print("Train:", train_df.shape, "| Validation:", val_df.shape, "| Test:", test_df.shape)


## 1. Data Quality Check — *owned by Member 1*
Confirms no missing values, empty strings, or duplicate reviews before any further processing.


In [ ]:
assert train_df.isna().sum().sum() == 0, "Missing values found — see Member 1's notebook"
assert train_df["text"].str.strip().eq("").sum() == 0, "Empty reviews found — see Member 1's notebook"
n_dupes = train_df.duplicated(subset=["text"]).sum()
print(f"Data quality check passed. Duplicate reviews found: {n_dupes}")


## 2. Text Cleaning & Normalization — *owned by Member 2*
Lowercases and normalizes punctuation/whitespace while preserving negation words.


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["clean_text"] = train_df["text"].apply(clean_text)
val_df["clean_text"] = val_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

train_df[["text", "clean_text"]].head(3)


## 3. Label Encoding — *owned by Member 3*
Encodes the sentiment category into the numeric form used by every downstream model.


In [ ]:
train_df["sentiment_category"] = train_df["label"].map({1: "positive", 0: "negative"})
le = LabelEncoder()
train_df["label_encoded"] = le.fit_transform(train_df["sentiment_category"])
print("Encoded classes:", list(le.classes_))


## 4. Outlier Check — *owned by Member 4*
Flags unusually short/long reviews by word count; group decision was to retain them (see
Member 4's notebook for the full justification).


In [ ]:
train_df["word_count"] = train_df["clean_text"].str.split().apply(len)
Q1, Q3 = train_df["word_count"].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
n_outliers = ((train_df["word_count"] < lower) | (train_df["word_count"] > upper)).sum()
print(f"Outliers flagged: {n_outliers} ({n_outliers/len(train_df):.1%}) — retained in the pipeline")


## 5. Vectorization & Scaling — *owned by Member 5*
Converts cleaned text into L2-normalized TF-IDF vectors.


In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), norm="l2")
X_tfidf = vectorizer.fit_transform(train_df["clean_text"])
print("TF-IDF matrix shape:", X_tfidf.shape)


## 6. Feature Selection & Dimensionality Reduction — *owned by Member 6*
Selects the most sentiment-informative terms and reduces to 2D for a visual sanity check.


In [ ]:
selector = SelectKBest(chi2, k=2000)
X_selected = selector.fit_transform(X_tfidf, train_df["label_encoded"])

svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_selected)

viz_df = pd.DataFrame({"x": X_2d[:, 0], "y": X_2d[:, 1],
                        "label": train_df["sentiment_category"]})
plt.figure(figsize=(7, 6))
sns.scatterplot(data=viz_df.sample(1500, random_state=1), x="x", y="y", hue="label",
                 palette={"negative": "#C1272D", "positive": "#2E9E6D"}, alpha=0.5, s=20)
plt.title("Group Pipeline Output: 2D Projection of Final Selected Features")
plt.show()

print(f"Final feature matrix shape after selection: {X_selected.shape}")


## 7. Final Processed Output
Save the fully processed dataset — this is what feeds Step 3's model training.


In [ ]:
processed = train_df[["text", "clean_text", "word_count", "sentiment_category", "label_encoded"]]
processed.to_csv("processed_train.csv", index=False)
print("Saved processed_train.csv —", processed.shape)
processed.head()


---
## Summary

| Stage | Owner | Technique |
|---|---|---|
| 1 | Member 1 | Missing data & duplicate detection |
| 2 | Member 2 | Text cleaning & normalization |
| 3 | Member 3 | Categorical (label) encoding |
| 4 | Member 4 | Outlier detection (review length) |
| 5 | Member 5 | Feature scaling (TF-IDF L2 normalization) |
| 6 | Member 6 | Feature selection & dimensionality reduction |

This processed output feeds directly into Step 3 (Model Design and Implementation) of the
main project.
